# Feature Engineering – DOGE/USDT Technical Indicators

El objetivo de este notebook es transformar el dataset bruto de mercado en un conjunto de datos enriquecido mediante técnicas de feature engineering orientadas al análisis financiero y al trading algorítmico.

Para ello, se generan indicadores técnicos, métricas de volatilidad, retornos, variables derivadas y etiquetas supervisadas que permitirán entrenar y evaluar modelos predictivos capaces de identificar patrones de comportamiento en el mercado de criptomonedas.

In [1]:
# ============================================================
# Imports and load dataset
# Carga el dataset bruto de velas de DOGE/USDT, convierte las columnas temporales a formato datetime, ordena los datos cronológicamente y prepara la ruta de salida para guardar el dataset enriquecido.
# ============================================================

import os
import numpy as np
import pandas as pd

RAW_PATH = "../data/raw/DOGEUSDT_5m_binance_2017_2026.csv"
OUTPUT_DIR = "../data/processed"
OUTPUT_FILE = "DOGEUSDT_5m_binance_2017_2026_features.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH, parse_dates=["open_time", "close_time"])
df = df.sort_values("open_time").reset_index(drop=True)

print("Loaded raw dataset")
print("Shape:", df.shape)
print("Date range:", df["open_time"].min(), "->", df["open_time"].max())

display(df.head())

Loaded raw dataset
Shape: (723409, 11)
Date range: 2019-07-05 12:00:00 -> 2026-05-23 11:10:00


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2019-07-05 12:00:00,0.004490,0.004600,0.003760,0.004350,281690734.0,2019-07-05 12:04:59.999,1.213789e+06,1841,131946467.0,572788.485474
1,2019-07-05 12:05:00,0.004340,0.004340,0.004000,0.004022,151285521.0,2019-07-05 12:09:59.999,6.310450e+05,963,41885000.0,173783.101331
2,2019-07-05 12:10:00,0.004022,0.004069,0.003800,0.003870,182657540.0,2019-07-05 12:14:59.999,7.079442e+05,1285,58720314.0,228693.805868
3,2019-07-05 12:15:00,0.003861,0.003940,0.003859,0.003929,73986015.0,2019-07-05 12:19:59.999,2.885708e+05,535,43864831.0,171156.394506
4,2019-07-05 12:20:00,0.003929,0.003964,0.003870,0.003877,67030342.0,2019-07-05 12:24:59.999,2.627509e+05,399,27619614.0,108776.180726


El dataset queda correctamente cargado y ordenado de forma cronológica, con más de 700.000 velas de 5 minutos entre julio de 2019 y mayo de 2026. Este volumen de datos permite trabajar con un histórico amplio, aunque también obliga a respetar estrictamente la estructura temporal para evitar mezclas entre pasado y futuro durante el modelado.

In [2]:
# ============================================================
# Technical indicators
# Esta celda convierte el precio bruto en un conjunto de variables más informativas para un modelo de machine learning. 
# El precio por sí solo suele ser pobre como predictor; los indicadores técnicos resumen tendencia, momentum, volatilidad y desviación respecto al comportamiento reciente.
# ============================================================

# Historical returns (past information only)
df["return_prev_1"] = df["close"].pct_change()
df["log_return_prev_1"] = np.log(df["close"] / df["close"].shift(1))

# Moving averages
df["sma_20"] = df["close"].rolling(window=20).mean()
df["ema_10"] = df["close"].ewm(span=10, adjust=False).mean()
df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()
df["ema_200"] = df["close"].ewm(span=200, adjust=False).mean()

# EMA ratios
df["ema10_ema50_ratio"] = df["ema_10"] / df["ema_50"]
df["ema50_ema200_ratio"] = df["ema_50"] / df["ema_200"]
df["sma20_ema50_ratio"] = df["sma_20"] / df["ema_50"]

# Rolling volatility: 1 hour = 12 candles of 5 minutes
df["volatility_1h"] = df["log_return_prev_1"].rolling(window=12).std()

# Z-score over 1 hour
rolling_mean_1h = df["close"].rolling(window=12).mean()
rolling_std_1h = df["close"].rolling(window=12).std()
df["zscore_close_1h"] = (df["close"] - rolling_mean_1h) / rolling_std_1h

# RSI 14
delta = df["close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

# MACD: 12, 26, 9
ema_12 = df["close"].ewm(span=12, adjust=False).mean()
ema_26 = df["close"].ewm(span=26, adjust=False).mean()

df["macd"] = ema_12 - ema_26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# Bollinger Bands: 20 periods, 2 std
bb_window = 20
bb_std = 2

bb_mid = df["close"].rolling(window=bb_window).mean()
bb_std_dev = df["close"].rolling(window=bb_window).std()

df["bb_mid"] = bb_mid
df["bb_upper"] = bb_mid + bb_std * bb_std_dev
df["bb_lower"] = bb_mid - bb_std * bb_std_dev
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_mid"]
df["bb_percent"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

# ATR 14
high_low = df["high"] - df["low"]
high_close_prev = (df["high"] - df["close"].shift(1)).abs()
low_close_prev = (df["low"] - df["close"].shift(1)).abs()

true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
df["atr_14"] = true_range.rolling(window=14).mean()

print("Features created")
print("Shape:", df.shape)

display(df.tail())

Features created
Shape: (723409, 32)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,rsi_14,macd,macd_signal,macd_hist,bb_mid,bb_upper,bb_lower,bb_width,bb_percent,atr_14
723404,2026-05-23 10:50:00,0.09957,0.09961,0.09951,0.09952,2019387.0,2026-05-23 10:54:59.999,201036.76574,672,1267731.0,...,52.873563,-0.000053,-0.000090,0.000038,0.099524,0.099679,0.099368,0.003117,0.488716,0.000115
723405,2026-05-23 10:55:00,0.09952,0.09954,0.09950,0.09952,284994.0,2026-05-23 10:59:59.999,28361.11493,286,211374.0,...,50.602410,-0.000051,-0.000082,0.000031,0.099520,0.099673,0.099368,0.003071,0.498364,0.000110
723406,2026-05-23 11:00:00,0.09952,0.09955,0.09946,0.09948,591135.0,2026-05-23 11:04:59.999,58808.86284,561,257920.0,...,51.219512,-0.000053,-0.000077,0.000024,0.099516,0.099669,0.099364,0.003067,0.380429,0.000110
723407,2026-05-23 11:05:00,0.09948,0.09962,0.09947,0.09954,1837323.0,2026-05-23 11:09:59.999,182909.11863,1442,1026953.0,...,58.536585,-0.000049,-0.000071,0.000022,0.099516,0.099668,0.099364,0.003059,0.578827,0.000114
723408,2026-05-23 11:10:00,0.09954,0.09961,0.09954,0.09956,286859.0,2026-05-23 11:14:59.999,28558.43106,459,182916.0,...,58.536585,-0.000043,-0.000065,0.000022,0.099515,0.099666,0.099364,0.003029,0.649286,0.000111


Tras esta transformación, el dataset pasa de contener únicamente variables de mercado básicas a incorporar indicadores que resumen distintas dimensiones del comportamiento del precio: tendencia, momentum, volatilidad y posición relativa dentro de rangos recientes. Todas estas variables se calculan usando información pasada o presente de cada vela, por lo que pueden utilizarse como entrada de modelos predictivos sin introducir información futura de forma accidental.

In [3]:
# ============================================================
# Supervised learning labels
# Esta celda define el problema supervisado: dado el estado del mercado en una vela, se intenta predecir si el precio subirá o bajará en distintos horizontes temporales. 
# Para el baseline inicial, up_1 o up_3 pueden funcionar como targets sencillos de clasificación binaria.
# ============================================================

horizons = [1, 3, 6, 12]

for h in horizons:
    df[f"future_close_{h}"] = df["close"].shift(-h)
    df[f"future_return_{h}"] = df[f"future_close_{h}"] / df["close"] - 1
    df[f"up_{h}"] = (df[f"future_return_{h}"] > 0).astype(int)

print("Labels created")
print("Available target columns:")

target_cols = [
    col for col in df.columns
    if col.startswith("future_close_")
    or col.startswith("future_return_")
    or col.startswith("up_")
]
print(target_cols)

# Safety check: old ambiguous target names should not exist anymore
old_target_names = [f"return_{h}" for h in horizons]
old_target_names_found = [col for col in old_target_names if col in df.columns]

if old_target_names_found:
    raise ValueError(
        f"Ambiguous old target columns found: {old_target_names_found}. "
        "Use future_return_* instead."
    )

print("Naming check passed: no ambiguous return_* target columns found.")

Labels created
Available target columns:
['future_close_1', 'future_return_1', 'up_1', 'future_close_3', 'future_return_3', 'up_3', 'future_close_6', 'future_return_6', 'up_6', 'future_close_12', 'future_return_12', 'up_12']
Naming check passed: no ambiguous return_* target columns found.


En esta fase se define explícitamente qué se quiere predecir. Para cada horizonte temporal se crean tanto el retorno futuro como una etiqueta binaria de subida o bajada. La separación entre return_prev_1, que representa información histórica, y future_return_*, que representa el objetivo futuro, es importante porque evita ambigüedades y reduce el riesgo de introducir data leakage en fases posteriores.

In [4]:
# ============================================================
# Support and resistance features
# Esta celda genera variables relacionadas con soportes y resistencias utilizando únicamente información histórica.
# El objetivo es aproximar conceptos clásicos del análisis técnico mediante métricas cuantitativas utilizables por modelos de machine learning.
# ============================================================

# Rolling range position
range_window = 288  # 24 hours of 5m candles

rolling_low = df["low"].rolling(window=range_window).min()
rolling_high = df["high"].rolling(window=range_window).max()

# Relative position of current price inside recent range
# 0 = near recent low, 1 = near recent high

df["price_position_in_recent_range"] = (
    (df["close"] - rolling_low)
    / (rolling_high - rolling_low)
)

# Dynamic support and resistance levels
support_window = 288

# Recent support/resistance approximations
df["recent_support"] = df["low"].rolling(window=support_window).min()
df["recent_resistance"] = df["high"].rolling(window=support_window).max()

# Distance to support/resistance
# Normalized by price to make values comparable across market regimes

df["dist_to_nearest_support"] = (
    (df["close"] - df["recent_support"])
    / df["close"]
)

df["dist_to_nearest_resistance"] = (
    (df["recent_resistance"] - df["close"])
    / df["close"]
)

# Touch counters
# Counts how many times price approached the support/resistance area recently

tolerance = 0.003  # 0.3%

df["near_support"] = (
    ((df["low"] - df["recent_support"]).abs() / df["close"] < tolerance)
).astype(int)

df["near_resistance"] = (
    ((df["high"] - df["recent_resistance"]).abs() / df["close"] < tolerance)
).astype(int)

# Strength estimation via rolling touch counts
strength_window = 288

df["support_strength"] = (
    df["near_support"]
    .rolling(window=strength_window)
    .sum()
)

df["resistance_strength"] = (
    df["near_resistance"]
    .rolling(window=strength_window)
    .sum()
)

# Combined touch count feature
df["touch_count_near_level"] = (
    df["support_strength"]
    + df["resistance_strength"]
)

print("Support/resistance features created")

support_cols = [
    "price_position_in_recent_range",
    "dist_to_nearest_support",
    "dist_to_nearest_resistance",
    "support_strength",
    "resistance_strength",
    "touch_count_near_level"
]

print(df[support_cols].describe())

Support/resistance features created
       price_position_in_recent_range  dist_to_nearest_support  \
count                   723122.000000            723122.000000   
mean                         0.498431                 0.039283   
std                          0.256459                 0.047533   
min                          0.000000                 0.000000   
25%                          0.283582                 0.014286   
50%                          0.494297                 0.026236   
75%                          0.711538                 0.046874   
max                          1.000000                 0.905706   

       dist_to_nearest_resistance  support_strength  resistance_strength  \
count               723122.000000     723122.000000        723122.000000   
mean                     0.040496         12.503286            14.918820   
std                      0.052066         18.042934            21.535127   
min                      0.000000          0.000000             0

Estas variables intentan traducir conceptos subjetivos del análisis técnico clásico a señales cuantificables. En lugar de asumir manualmente dónde existen soportes o resistencias, el sistema utiliza ventanas móviles y densidad de “toques” recientes para estimar zonas relevantes del precio.

Aunque este enfoque sigue siendo relativamente simple frente a técnicas más avanzadas de clustering o detección estructural de niveles, permite incorporar contexto de mercado adicional que no capturan indicadores tradicionales como RSI o MACD. Además, todas las variables se calculan exclusivamente con información histórica, manteniendo la coherencia temporal necesaria para el entrenamiento de modelos predictivos.

In [5]:
# ============================================================
# Clean and save processed dataset
# Elimina las filas con valores nulos generados por los cálculos rolling, EMAs iniciales y desplazamientos futuros de los labels. Después guarda el dataset final en la carpeta data/processed.
# ============================================================

print("Shape before dropna:", df.shape)
print("Missing values before dropna:", df.isna().sum().sum())

df_features = df.dropna().reset_index(drop=True)

print("Shape after dropna:", df_features.shape)
print("Missing values after dropna:", df_features.isna().sum().sum())

df_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved processed dataset to: {OUTPUT_PATH}")

display(df_features.head())
display(df_features.tail())

Shape before dropna: (723409, 54)
Missing values before dropna: 2564
Shape after dropna: (723080, 54)
Missing values after dropna: 0
Saved processed dataset to: ../data/processed\DOGEUSDT_5m_binance_2017_2026_features.csv


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,price_position_in_recent_range,recent_support,recent_resistance,dist_to_nearest_support,dist_to_nearest_resistance,near_support,near_resistance,support_strength,resistance_strength,touch_count_near_level
0,2019-07-06 11:55:00,0.003630,0.003644,0.003630,0.003635,472036.0,2019-07-06 11:59:59.999,1718.044681,24,405943.0,...,0.081048,0.00355,0.004600,0.023411,0.265440,0,0,0.0,0.0,0.0
1,2019-07-06 12:00:00,0.003632,0.003638,0.003629,0.003637,799853.0,2019-07-06 12:04:59.999,2908.008956,18,292138.0,...,0.109846,0.00355,0.004340,0.023867,0.193412,0,0,0.0,0.0,0.0
2,2019-07-06 12:05:00,0.003634,0.003637,0.003625,0.003637,2176492.0,2019-07-06 12:09:59.999,7909.776304,42,1028365.0,...,0.166956,0.00355,0.004069,0.023813,0.118820,0,0,0.0,0.0,0.0
3,2019-07-06 12:10:00,0.003636,0.003636,0.003620,0.003620,618720.0,2019-07-06 12:14:59.999,2240.639784,16,33792.0,...,0.168759,0.00355,0.003964,0.019310,0.095113,0,0,0.0,0.0,0.0
4,2019-07-06 12:15:00,0.003621,0.003627,0.003610,0.003611,1808302.0,2019-07-06 12:19:59.999,6538.506892,30,450314.0,...,0.147272,0.00355,0.003964,0.016893,0.097812,0,0,0.0,0.0,0.0


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,price_position_in_recent_range,recent_support,recent_resistance,dist_to_nearest_support,dist_to_nearest_resistance,near_support,near_resistance,support_strength,resistance_strength,touch_count_near_level
723075,2026-05-23 09:50:00,0.09950,0.09952,0.09943,0.09946,1679336.0,2026-05-23 09:54:59.999,167050.11271,1127,783613.0,...,0.221987,0.09736,0.10682,0.021114,0.074000,0,0,34.0,18.0,52.0
723076,2026-05-23 09:55:00,0.09947,0.09948,0.09938,0.09940,1066829.0,2026-05-23 09:59:59.999,106080.10005,780,384552.0,...,0.215645,0.09736,0.10682,0.020523,0.074648,0,0,34.0,18.0,52.0
723077,2026-05-23 10:00:00,0.09941,0.09947,0.09937,0.09942,815385.0,2026-05-23 10:04:59.999,81075.92691,997,222029.0,...,0.217759,0.09736,0.10682,0.020720,0.074432,0,0,34.0,18.0,52.0
723078,2026-05-23 10:05:00,0.09942,0.09948,0.09937,0.09946,3255114.0,2026-05-23 10:09:59.999,323681.19633,1232,2459145.0,...,0.221987,0.09736,0.10682,0.021114,0.074000,0,0,34.0,18.0,52.0
723079,2026-05-23 10:10:00,0.09945,0.09955,0.09945,0.09955,312988.0,2026-05-23 10:14:59.999,31143.08415,922,176706.0,...,0.231501,0.09736,0.10682,0.021999,0.073029,0,0,34.0,18.0,52.0


La limpieza elimina únicamente una fracción muy pequeña del dataset, correspondiente a las primeras filas afectadas por ventanas móviles y a las últimas filas necesarias para calcular los targets futuros. El dataset final conserva 723.348 registros y queda sin valores nulos, por lo que ya puede utilizarse en las siguientes fases del proyecto: análisis exploratorio, entrenamiento de modelos y backtesting.

# Conclusión

Este notebook convierte el histórico bruto de DOGE/USDT en un dataset preparado para machine learning supervisado. El proceso añade indicadores técnicos, variables derivadas y targets futuros claramente diferenciados, manteniendo la lógica temporal necesaria en problemas de series temporales financieras.

El punto más importante no es sólo haber generado más columnas, sino haber separado correctamente las variables explicativas, basadas en información pasada, de las etiquetas que representan el comportamiento futuro. Esta distinción será clave para que los resultados de los modelos y de los backtests posteriores sean interpretables y no estén contaminados por información que no existiría en un escenario real de trading.